# 🚀 Google Colab × Antigravity 智能联动实验室

本 Notebook 包含：
1. **Cloudflare Quick Tunnel 远程 SSH 穿透**：一键生成终端直连通道，让本地助手可直接操控云端 GPU。
2. **经典深度学习与金融量化实验**：PyTorch GPU 张量加速测试、多周期量价特征工程与 LightGBM 拟合。
3. **云端音视频处理套件测试**：FFmpeg 环境与 AI 语音转字幕。

## 步骤 1：启动 Cloudflare Tunnel 与 SSH 服务
点击运行下方代码，等待数秒即可看到生成的 SSH 直连命令。

In [ ]:
# 1. 一键启动稳定版 Colab-SSH (基于 Cloudflare Tunnel)
!pip install colab_ssh --upgrade -q
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="colab123")


## 步骤 2：经典机器学习与金融多周期量价实验
测试 GPU 矩阵算力以及基于多周期形态（如均线翻转斜率与量能变化）的分类与夏普回测。

In [ ]:
import torch, time
import numpy as np

# 1. GPU 检测
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 显卡型号:", torch.cuda.get_device_name(0))
    print(f"GPU 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# 2. 深度学习神经网络梯度反向传播
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4096, 128, device=device)
y = torch.randn(4096, 1, device=device)
model = torch.nn.Sequential(
    torch.nn.Linear(128, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 1)
).to(device)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
for _ in range(100):
    loss = loss_fn(model(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"✅ 深度学习 100 Epochs 训练完成，耗时: {time.time()-t0:.4f} 秒")

# 3. 金融多周期量价形态与 GBDT 模拟
try:
    import lightgbm as lgb
    np.random.seed(42)
    n = 10000
    ma_slope = np.random.randn(n)        # 60分均线斜率
    vol_ratio = np.random.exponential(1.0, n)  # 突破放量倍数
    bias = np.random.randn(n)            # 15分乖离率
    y = ((0.12 * ma_slope + 0.08 * vol_ratio + np.random.normal(0, 0.5, n)) > 0).astype(int)
    X = np.column_stack([ma_slope, vol_ratio, bias])
    clf = lgb.LGBMClassifier(n_estimators=50, max_depth=3, verbose=-1)
    clf.fit(X[:8000], y[:8000])
    acc = np.mean(clf.predict(X[8000:]) == y[8000:])
    print(f"✅ 金融 LightGBM 因子重要性: {clf.feature_importances_} | 测试集准确率: {acc*100:.2f}%")
except Exception as e:
    print("LightGBM 运行跳过:", e)

## 步骤 3：微软 Qlib 工业级 AI 量化金融平台实验
使用真实 A 股历史行情数据、Alpha158 量价因子库进行时序特征提取与模型训练，并评估真实 IC / Rank IC。

In [ ]:
# ==============================================================================
# 微软 Qlib 工业级 AI 量化金融平台实验 (通过 uv 调度 Python 3.11 环境)
# ==============================================================================

# 1. 利用 uv 快速部署 Python 3.11 兼容环境与 Qlib 预编译轮子
!pip install uv -q
!uv venv --python 3.11 /content/qlib_env -q
!uv pip install --python /content/qlib_env/bin/python pyqlib tables lightgbm requests tqdm -q

# 2. 极速下载微软预处理好的 A 股日频真实行情数据集 (~196MB)
!mkdir -p ~/.qlib/qlib_data/cn_data
!wget -q -N https://github.com/SunsetWolf/qlib_dataset/releases/download/v2/qlib_data_cn_1d_latest.zip -O /tmp/cn_data.zip
!unzip -q -o /tmp/cn_data.zip -d ~/.qlib/qlib_data/cn_data/
!rm -f /tmp/cn_data.zip
print("✅ A 股全量历史行情数据已同步至 ~/.qlib/qlib_data/cn_data")

# 3. 生成并运行完整的 Qlib A 股 Alpha158 实盘因子回测
script_cn = r'''
import qlib
from qlib.constant import REG_CN
from qlib.data import D
import pandas as pd
import numpy as np

# 初始化国内数据引擎
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region=REG_CN)
print("=" * 60)
print("✅ Qlib 引擎与真实 A 股数据初始化成功！")

# 获取沪深300成分股样本
instruments = D.instruments(market="csi300")
stock_list = D.list_instruments(instruments=instruments, as_list=True)
print(f"📊 沪深300 (CSI300) 股票样本总数: {len(stock_list)}，示例: {stock_list[:6]}")

# 读取贵州茅台与宁德时代真实行情
df = D.features(
    instruments=["SH600519", "SZ300750"],
    fields=["$close", "$open", "$high", "$low", "$volume", "$factor"],
    start_time="2020-01-01",
    end_time="2020-06-30"
)
print("
📈 真实行情数据样例 (含收盘价、成交量与复权因子)：")
print(df.head(6))

# 运行微软 Alpha158 真实因子流水线与模型评测
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.utils import init_instance_by_config

print("
🔄 正在构建 Alpha158 因子库 (动量、均线斜率、量价背离等 158 维技术特征)...")
dataset_config = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "Alpha158",
            "module_path": "qlib.contrib.data.handler",
            "kwargs": {
                "start_time": "2019-01-01",
                "end_time": "2020-06-30",
                "fit_start_time": "2019-01-01",
                "fit_end_time": "2019-12-31",
                "instruments": "csi300",
            },
        },
        "segments": {
            "train": ("2019-01-01", "2019-12-31"),
            "test": ("2020-01-01", "2020-06-30"),
        },
    },
}

dataset = init_instance_by_config(dataset_config)
df_train = dataset.prepare("train")
df_test = dataset.prepare("test")
print(f"✅ 训练集样本规模: {len(df_train)}, 测试集样本规模: {len(df_test)}")
print(f"✅ 特征维度数: {df_train.shape[1]} (Alpha158 全部因子就绪)")

# 训练 LightGBM 预测模型
model = LGBModel(loss="mse", n_estimators=60, learning_rate=0.08, verbose=-1)
print("🚀 正在拟合 LightGBM 模型...")
model.fit(dataset)

# 预测与计算 IC 表现
pred = model.predict(dataset)
label = dataset.prepare("test", col_set="label")
pred_label = pd.concat([pred, label], axis=1).dropna()

ic = pred_label.groupby(level="datetime").apply(lambda x: x.iloc[:, 0].corr(x.iloc[:, 1])).mean()
rank_ic = pred_label.groupby(level="datetime").apply(lambda x: x.iloc[:, 0].corr(x.iloc[:, 1], method="spearman")).mean()

print("=" * 60)
print("🏆 微软 Qlib A 股 Alpha158 实盘历史回测指标:")
print(f"  测试集样本外信息系数 (Normal IC) : {ic:.4f}")
print(f"  测试集样本外秩相关系数 (Rank IC)   : {rank_ic:.4f}")
if rank_ic > 0.02:
    print("  💡 评语: Rank IC 显著为正，具备实盘选股超额收益能力！")
print("=" * 60)
'''

with open("/content/run_qlib_cn.py", "w", encoding="utf-8") as f:
    f.write(script_cn)

!/content/qlib_env/bin/python /content/run_qlib_cn.py


## 步骤 4：微软 Qlib 美股市场实验 (官方美股数据集 + 科技巨头 Alpha158 + LightGBM 收益率预测)
1. **自动拉取官方美股行情包** (`region="us"`, 标普500/纳斯达克100成分股及历史日频行情)。
2. **构建美股专属 Alpha158 因子库**：提取动量、量价背离、均线多头排列等 158 维量化技术因子。
3. **训练 GBDT 机器学习模型**：使用微软 LightGBM 算法拟合历史规律，预测美股科技龙头（AAPL, MSFT, NVDA, TSLA, AMZN, GOOGL 等）超额收益。
4. **样本外回测评估**：输出量化机构核心指标 Normal IC、Rank IC 及 Rank ICIR。


In [ ]:
# ==============================================================================
# 微软 Qlib 美股量化实验：拉取官方美股数据集 + 科技巨头 Alpha158 因子预测
# ==============================================================================

# 1. 下载微软官方美股数据集 (~429MB) 并解压到 ~/.qlib/qlib_data/us_data
!mkdir -p ~/.qlib/qlib_data/us_data
!echo "📥 正在极速拉取微软 Qlib 官方美股数据集 (约 429MB)..."
!wget -q -N https://github.com/SunsetWolf/qlib_dataset/releases/download/v2/qlib_data_us_1d_latest.zip -O /tmp/us_data.zip
!unzip -q -o /tmp/us_data.zip -d ~/.qlib/qlib_data/us_data/
!rm -f /tmp/us_data.zip
!echo "✅ 美股官方数据集解压就绪！"

# 2. 生成美股 Alpha158 + LightGBM 预测与评估脚本
script_us = r'''
import qlib
from qlib.constant import REG_US
from qlib.data import D
import pandas as pd
import numpy as np

# 初始化美股数据引擎
qlib.init(provider_uri="~/.qlib/qlib_data/us_data", region=REG_US)
print("=" * 65)
print("🇺🇸 Microsoft Qlib 美股量化金融引擎初始化成功！")

# 1. 检查美股股票池与行情
sp500 = D.instruments(market="sp500")
sp500_list = D.list_instruments(instruments=sp500, as_list=True)
nasdaq = D.instruments(market="nasdaq100")
nasdaq_list = D.list_instruments(instruments=nasdaq, as_list=True)
print(f"📊 标普500 (S&P 500) 成分股总数: {len(sp500_list)}")
print(f"📊 纳斯达克100 (NASDAQ 100) 成分股总数: {len(nasdaq_list)}")

# 选取典型美股科技巨头标的池 (Magnificent 7)
tech_candidates = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
all_us_stocks = set(sp500_list + nasdaq_list)
tech_giants = [s for s in tech_candidates if s in all_us_stocks]
print(f"🎯 选定美股科技龙头回测标的池: {tech_giants}")

# 查看苹果与微软行情特征
df_sample = D.features(
    instruments=tech_giants[:2],
    fields=["$close", "$open", "$high", "$low", "$volume", "$factor"],
    start_time="2018-01-01",
    end_time="2018-06-30"
)
print("
📈 科技龙头真实历史行情样例 (前 6 行)：")
print(df_sample.head(6))

# 2. 构建美股版 Alpha158 因子流水线
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.utils import init_instance_by_config

print("
🔄 正在构建美股 Alpha158 因子工程 (计算 158 维高阶量价时序特征)...")
dataset_config = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "Alpha158",
            "module_path": "qlib.contrib.data.handler",
            "kwargs": {
                "start_time": "2015-01-01",
                "end_time": "2020-08-31",
                "fit_start_time": "2015-01-01",
                "fit_end_time": "2018-12-31",
                "instruments": tech_giants,
            },
        },
        "segments": {
            "train": ("2015-01-01", "2018-12-31"),
            "test": ("2019-01-01", "2020-08-31"),
        },
    },
}

dataset = init_instance_by_config(dataset_config)
df_train = dataset.prepare("train")
df_test = dataset.prepare("test")
print(f"✅ 训练集样本数 (2015-2018): {len(df_train)}, 测试集样本数 (2019-2020): {len(df_test)}")
print(f"✅ 特征维度数: {df_train.shape[1]} (Alpha158 美股因子集全部就绪)")

# 3. 拟合微软 LightGBM 预测模型
print("
🚀 正在使用 LightGBM GBDT 拟合美股量价非线性规律...")
model = LGBModel(
    loss="mse",
    n_estimators=70,
    learning_rate=0.06,
    max_depth=4,
    num_leaves=16,
    verbose=-1
)
model.fit(dataset)

# 4. 样本外回测评估与 IC / Rank IC 计算
pred = model.predict(dataset)
label = dataset.prepare("test", col_set="label")
pred_label = pd.concat([pred, label], axis=1).dropna()
pred_label.columns = ["pred", "label"]

ic = pred_label.groupby(level="datetime").apply(lambda x: x["pred"].corr(x["label"]))
rank_ic = pred_label.groupby(level="datetime").apply(lambda x: x["pred"].corr(x["label"], method="spearman"))

mean_ic = ic.mean()
mean_rank_ic = rank_ic.mean()
icir = mean_ic / (ic.std() + 1e-8)
rank_icir = mean_rank_ic / (rank_ic.std() + 1e-8)

print("=" * 65)
print("🏆 微软 Qlib 美股科技龙头 Alpha158 + LightGBM 回测评测报告:")
print(f"  测试集样本外预测信息系数 (Normal IC) : {mean_ic:.4f}")
print(f"  测试集样本外秩相关系数 (Rank IC)   : {mean_rank_ic:.4f}")
print(f"  信息比率 (ICIR)                    : {icir:.4f}")
print(f"  秩信息比率 (Rank ICIR)             : {rank_icir:.4f}")
if mean_rank_ic > 0.02:
    print("  💡 评语: Rank IC > 0.02，模型对美股科技龙头超额收益具有显著预测能力！")
print("=" * 65)
'''

with open("/content/run_qlib_us.py", "w", encoding="utf-8") as f:
    f.write(script_us)

# 3. 执行美股实验并输出完整评估报告
!/content/qlib_env/bin/python /content/run_qlib_us.py
